# AutoEIT Transcription Pipeline Tutorial

This notebook demonstrates the complete transcription pipeline for processing learner Spanish speech.

**Goals:**
- Load and preprocess audio files
- Transcribe using Whisper-large-v3
- Post-process transcriptions
- Evaluate against reference transcriptions
- Measure WER (Word Error Rate) and agreement metrics

## Section 1: Import Required Libraries

In [ ]:
import sys
import logging
from pathlib import Path
import json

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / "src"))

# Core libraries
import numpy as np
import pandas as pd
from pathlib import Path

# Audio processing
import torchaudio
import librosa
import matplotlib.pyplot as plt
import seaborn as sns

# AutoEIT modules
from transcription.pipeline import TranscriptionPipeline
from transcription.preprocessor import AudioPreprocessor
from transcription.postprocessor import PostProcessor
from transcription.evaluator import TranscriptionEvaluator
from utils.metrics import word_error_rate, percent_agreement

# Logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Display settings
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

print("✓ All libraries imported successfully!")

## Section 2: Load and Explore Audio Files

Let's start by loading sample audio files and examining their properties.

In [ ]:
# Define paths
AUDIO_DIR = Path("../data/sample")
MANIFEST_PATH = AUDIO_DIR / "train_manifest.json"

# Load manifest
with open(MANIFEST_PATH, "r") as f:
    manifest = json.load(f)

# Create sample dataframe
samples_df = pd.DataFrame(manifest)
print(f"Loaded {len(samples_df)} sample sentences:")
print(samples_df.head(10))

## Section 3: Audio Preprocessing Pipeline

The preprocessing step handles:
- Resampling to 16 kHz (Whisper standard)
- Conversion to mono
- Noise reduction via spectral gating
- RMS normalization
- Silence trimming and segmentation

In [ ]:
# Initialize preprocessor
preprocessor = AudioPreprocessor()
print(f"Preprocessor config:\n{preprocessor.config}")

# Example: Show preprocessing on first sample (if audio exists)
# Note: For demo purposes, we'll show the configuration
print("\nPreprocessing workflow:")
print("1. Load audio (any format: .wav, .mp3, .flac)")
print("2. Resample to 16 kHz")
print("3. Convert stereo → mono")
print("4. Apply spectral noise gating")
print("5. RMS normalize")
print("6. Segment on silence boundaries")
print("7. Trim silence from edges")
print("8. Filter segments by duration (0.5s - 30s)")

## Section 4: Transcription Pipeline

The transcription uses fine-tuned Whisper-large-v3 with LoRA for learner speech.

In [ ]:
# Example ASR results for demonstration
# In a real scenario, these would come from the Whisper model
demo_results = {
    "hypothesis": [
        "yo fui al mercado ayer con mi madre",
        "mi hermana estudia la universidad",
        "nosotros viajamos españa",
        "el gato esta durmiendo",
    ],
    "reference": [
        "yo fui al mercado ayer con mi madre",
        "mi hermana estudia en la universidad",
        "nosotros viajamos a españa",
        "el gato está durmiendo",
    ],
}

results_df = pd.DataFrame(demo_results)
print("Transcription Results:")
print(results_df.to_string())

## Section 5: Post-Processing & Error Correction

Post-processing corrects systematic ASR errors in learner speech:
- Remove disfluencies (hesitation markers)
- Fix phonological transfer errors
- Normalize punctuation

In [ ]:
# Initialize post-processor
postprocessor = PostProcessor()

# Example: Apply post-processing to noisy ASR output
raw_output = "um este yo fui al mercado ayer con mi— con mi madre"
cleaned = postprocessor.process(raw_output)
print(f"Raw ASR:  {raw_output}")
print(f"Cleaned:  {cleaned}")

# Show explanation
explanation = postprocessor.explain(raw_output, cleaned)
print(f"\nChanges made:")
print(f"  Removed: {explanation['removed']}")
print(f"  Added:   {explanation['added']}")

## Section 6: Evaluation & Metrics

Measure transcription quality against human reference transcriptions.

**Goal:** ≤10% WER (Word Error Rate) → ~90% agreement with human transcribers

In [ ]:
# Initialize evaluator with target WER of 10%
evaluator = TranscriptionEvaluator(target_wer=0.10)

# Evaluate the demo results
eval_result = evaluator._aggregate([
    evaluator._evaluate_sample(
        f"sample_{i:03d}.wav",
        ref,
        hyp,
    )
    for i, (ref, hyp) in enumerate(
        zip(results_df["reference"], results_df["hypothesis"])
    )
])

# Print evaluation summary
evaluator.print_summary(eval_result)

# Detailed results
print("\n" + "=" * 60)
print("Detailed Per-Sample Results:")
print("=" * 60)
for sample in eval_result.samples:
    print(f"\n{sample.audio_file}:")
    print(f"  Reference: {sample.reference}")
    print(f"  Hypothesis: {sample.hypothesis}")
    print(f"  WER: {sample.wer:.1%}")
    print(f"  Exact Match: {sample.exact_match}")
    print(f"  Errors: {sample.errors}")

## Section 7: Visualization of Results

In [ ]:
# Extract WER values for visualization
wer_values = [sample.wer for sample in eval_result.samples]
files = [sample.audio_file for sample in eval_result.samples]

# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot of WER by sample
axes[0].bar(range(len(files)), [w * 100 for w in wer_values], color="steelblue")
axes[0].axhline(y=10, color="red", linestyle="--", label="Target WER (10%)")
axes[0].set_xlabel("Sample")
axes[0].set_ylabel("WER (%)")
axes[0].set_title("Word Error Rate by Sample")
axes[0].set_xticks(range(len(files)))
axes[0].set_xticklabels([f"S{i+1}" for i in range(len(files))], rotation=0)
axes[0].legend()
axes[0].grid(alpha=0.3)

# Summary metrics
metrics = {
    "Avg WER": eval_result.avg_wer * 100,
    "Avg CER": eval_result.avg_cer * 100,
    "Exact Match": eval_result.exact_match_rate * 100,
}
colors = ["green" if eval_result.meets_target else "orange"]
axes[1].barh(list(metrics.keys()), list(metrics.values()), color="skyblue")
axes[1].set_xlabel("Percentage (%)")
axes[1].set_title("Overall Metrics Summary")
axes[1].grid(alpha=0.3, axis="x")

plt.tight_layout()
plt.show()

print(f"\n✓ Meets Target: {eval_result.meets_target}")

## Section 8: Export Results

Save transcription results and evaluation metrics to files.

In [ ]:
# Create output directory
output_dir = Path("../results")
output_dir.mkdir(exist_ok=True)

# 1. Save transcription results as CSV
output_csv = output_dir / "transcriptions.csv"
results_df.to_csv(output_csv, index=False)
print(f"✓ Saved transcriptions to {output_csv}")

# 2. Save detailed evaluation results as JSON
evaluator.save_report(eval_result, output_dir / "eval_report.json")
print(f"✓ Saved evaluation report to {output_dir / 'eval_report.json'}")

# 3. Save summary metrics as JSON
summary = {
    "total_samples": eval_result.total_samples,
    "avg_wer": float(eval_result.avg_wer),
    "avg_cer": float(eval_result.avg_cer),
    "exact_match_rate": float(eval_result.exact_match_rate),
    "meets_target": eval_result.meets_target,
    "target_wer": evaluator.target_wer,
}

with open(output_dir / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(f"✓ Saved summary to {output_dir / 'summary.json'}")

# Display summary
print("\n" + "=" * 60)
print("EXPORT COMPLETE")
print("=" * 60)
print(f"Output directory: {output_dir}")
print(f"Files created:")
print(f"  - transcriptions.csv")
print(f"  - eval_report.json")
print(f"  - summary.json")